# r_tower — Two-tower retrieval с logQ-коррекцией

**Статьи:**
- Covington, Adams, Sargin (2016) — *Deep Neural Networks for YouTube Recommendations*. Candidate generation как extreme multiclass: user-башня строится из **среднего эмбеддингов просмотренных айтемов** («watch vector»), скор = dot product, обучение — sampled softmax.
- Yi et al. (2019, RecSys) — *Sampling-Bias-Corrected Neural Modeling for Large Corpus Item Recommendations*. **Ключевой вклад, который мы реализуем:** in-batch негативы делают популярные айтемы негативом слишком часто и смещают softmax. Лечится **logQ-коррекцией** — из каждого in-batch логита вычитается $\log Q(\text{item})$ (вероятность попасть в батч, оценённая стриминговым частотным оценщиком) ДО softmax.

**Узел дерева:** RecSys → retrieval → two-tower / neural candidate generation

**Что строим и обучаем:** двухбашенную retrieval-модель на MovieLens-100k (implicit feedback, leave-one-out по времени), обучаем in-batch sampled softmax, меряем Recall@K (== HitRate@K — один held-out таргет на юзера). Сравниваем три системы: popularity-бейзлайн, two-tower без коррекции (абляция), two-tower с logQ-коррекцией.

**Компьют:** CPU, 3 эпохи, несколько минут. Датасет (~5 МБ) качается в setup-ячейке.

**Метрика — из поля candidate generation:** Recall@10 и Recall@50 на held-out последнем взаимодействии (Covington меряет ровно «попал ли следующий просмотр в top-N»).

**Parts:**
1. Watch vector — user-башня как среднее истории (Covington).
2. In-batch sampled softmax + logQ-коррекция (вклад Yi 2019).
3. Стриминговый частотный оценщик $Q$ (Algorithm 1 Yi 2019).
4. Training loop — пишешь сама, обучаешь обе модели.
5. Retrieval + Recall@K — применяем метод, замыкаем цикл на метрике.
6. Абляция: коррекция vs без неё vs popularity — пощупать фишку.


## Эксперименты-первыми (заполни ДО чтения метода)

Открой Yi et al. (2019), раздел экспериментов, и ответь своими словами — это якорь на весь разбор.

> **(a) Что результаты реально показали против обещаний абстракта?** (на каком корпусе, насколько logQ-коррекция сдвинула метрику, где эффект был сильнее всего)
>
> …

> **(b) Самый слабый эксперимент / чем бы фальсифицировала вклад?** (что если негативы не in-batch, а из uniform-сэмплера — нужна ли тогда коррекция вообще?)
>
> …

> **(c) Где сломалось / чего НЕ протестировали?** (как ведёт себя стриминговый оценщик на холодных айтемах; чувствительность к $\alpha$)
>
> …


## Зачем это

Retrieval-стадия рекомендера должна из миллионов айтемов за миллисекунды отобрать сотни кандидатов. Полный softmax по всему каталогу на каждом шаге обучения невозможен, поэтому считают softmax только по айтемам **внутри батча**: позитив каждого юзера служит негативом всем остальным юзерам батча. Дёшево и параллельно — но есть подвох.

In-batch негативы сэмплируются пропорционально частоте: блокбастер попадает в батч (а значит, в негативы) на порядки чаще нишевого фильма. Sampled softmax в таком виде оценивает не тот full-softmax, который нам нужен, — он систематически **штрафует популярные айтемы**, потому что они слишком часто оказываются «неправильным ответом». Модель недооценивает популярное и переоценивает хвост.

**Фишка Yi 2019 — logQ-коррекция.** Если из логита айтема вычесть $\log Q(\text{item})$ — лог-вероятность того, что этот айтем вообще попал в батч, — то смещённая in-batch оценка становится несмещённой оценкой full-softmax. Это та же поправка, что в noise-contrastive / sampled-softmax теории (Bengio & Senécal), но $Q$ здесь не задано аналитически (батчи приходят из живого распределения трафика), поэтому его **оценивают на лету** стриминговым алгоритмом. Преимущество над альтернативой (uniform-сэмплинг негативов из всего каталога): in-batch негативы бесплатны и уже на GPU, а коррекция возвращает им несмещённость — получаем и скорость, и корректный градиент.

Почему MovieLens-100k: компактный публичный implicit-feedback бенчмарк, на котором two-tower рутинно sanity-чекают. Leave-one-out по таймстемпу — стандартный протокол top-K retrieval для секвенциальных рекомендеров: последнее взаимодействие юзера прячем как таргет, остальное — история.


## Теория (Фейнман)

**Задача.** Юзер $u$ с историей просмотров $H_u$; хотим скорить все айтемы $y$ и вернуть top-K. Two-tower: user-башня $\phi(u)\in\mathbb{R}^d$, item-башня $\psi(y)\in\mathbb{R}^d$, скор $s(u,y)=\langle\phi(u),\psi(y)\rangle$. Обе башни L2-нормируются (косинус) и логиты делятся на температуру $\tau$.

**User-башня (Covington).** Никакого id-эмбеддинга юзера. Представление юзера — функция его истории: усреднить эмбеддинги просмотренных айтемов («watch vector») и пропустить через MLP:
$$\phi(u)=\mathrm{MLP}\!\Big(\tfrac{1}{|H_u|}\sum_{y\in H_u}\psi(y)\Big).$$
Так модель обобщается на новых юзеров и на новые истории без переобучения.

**Full softmax (чего хотим).** Вероятность следующего айтема $y$ для юзера $u$:
$$P(y\mid u)=\frac{\exp s(u,y)}{\sum_{y'\in\mathcal{Y}}\exp s(u,y')}.$$
Знаменатель по всему каталогу $\mathcal{Y}$ недосчитываем.

**In-batch sampled softmax.** В батче $B$ айтемов; для юзера $i$ знаменатель берём только по айтемам батча:
$$P_B(j\mid i)=\frac{\exp s_{ij}}{\sum_{k\in B}\exp s_{ik}},\qquad s_{ij}=\langle\phi(u_i),\psi(y_j)\rangle.$$
Правильный класс строки $i$ — диагональ $j=i$. Но айтемы попадают в $B$ пропорционально $Q(y)$, поэтому $P_B$ — **смещённая** оценка $P$.

**logQ-коррекция (вклад Yi 2019).** Скорректированный логит:
$$s^{c}_{ij}=s_{ij}-\log Q(y_j).$$
Подстановка в softmax по $B$ даёт несмещённую (Monte-Carlo) оценку full-softmax: вычитание $\log Q$ ровно компенсирует, что популярный $y_j$ попадает в знаменатель чаще. Лосс — кросс-энтропия с диагональными метками.

**Оценка $Q$ (Algorithm 1).** $Q(y)$ не дано — оцениваем по среднему числу шагов между двумя появлениями $y$ в потоке батчей. Для айтема $y$: $A[y]$ — последний шаг, когда видели $y$; $B[y]$ — бегущее среднее интервала между появлениями. На шаге $t$, увидев $y$:
$$B[y]\leftarrow(1-\alpha)\,B[y]+\alpha\,(t-A[y]),\qquad A[y]\leftarrow t,\qquad \hat Q(y)=1/B[y].$$
Часто появляется → маленький интервал → большой $\hat Q$ → большое $\log Q$ → сильнее штраф в логите. Логика замыкается.

**Метрика.** Recall@K с одним held-out таргетом — это HitRate@K: 1, если спрятанный айтем попал в top-K, иначе 0. Уже виденные айтемы из истории при ретриве маскируем.

*Где у меня дыра в понимании:* …

❓ **Inline Question 1.** Почему именно **вычитание** $\log Q$ (а не, скажем, деление логита на $Q$ или умножение скора)? Подсказка: что коррекция делает с логарифмом софтмакс-веса $\exp(s_{ij})$ в знаменателе?

**Ваш ответ:** …


In [ ]:
# --- setup (готово, просто запусти) ---
# pip install torch numpy
import os, io, zipfile, urllib.request
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cpu"

DATA_DIR = os.path.join(os.getcwd(), "ml-100k")
URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"

def load_ratings():
    path = os.path.join(DATA_DIR, "u.data")
    if not os.path.exists(path):
        print("downloading MovieLens-100k ...")
        raw = urllib.request.urlopen(URL, timeout=60).read()
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            z.extractall(os.getcwd())
    rows = []
    with open(path) as f:
        for line in f:
            u, i, r, t = line.split("\t")
            rows.append((int(u), int(i), int(r), int(t)))
    return rows  # (user, item, rating, ts)

def build_dataset():
    rows = load_ratings()
    rows = [(u, i, t) for (u, i, r, t) in rows if r >= 4]   # implicit: rating>=4 = "watch"
    users = sorted({u for u, _, _ in rows}); items = sorted({i for _, i, _ in rows})
    uid = {u: k for k, u in enumerate(users)}; iid = {i: k for k, i in enumerate(items)}
    n_items = len(items)
    by_user = defaultdict(list)
    for u, i, t in rows:
        by_user[uid[u]].append((t, iid[i]))
    train_pairs = []; test = {}; item_pop = np.zeros(n_items)
    for u, seq in by_user.items():
        seq.sort()
        items_seq = [i for _, i in seq]
        if len(items_seq) < 5:
            continue
        test_hist, test_target = items_seq[:-1], items_seq[-1]   # leave-one-out by time
        test[u] = (test_hist, test_target)
        for k in range(1, len(test_hist)):                       # predict each item from its prefix
            train_pairs.append((u, items_seq[:k], items_seq[k]))
            item_pop[items_seq[k]] += 1.0
    return train_pairs, test, n_items, item_pop

def collate(batch):
    hists = [h for (_, h, _) in batch]
    targets = torch.tensor([t for (_, _, t) in batch], dtype=torch.long)
    L = max(len(h) for h in hists)
    padded = torch.zeros(len(batch), L, dtype=torch.long)
    lens = torch.zeros(len(batch), dtype=torch.long)
    for r, h in enumerate(hists):
        padded[r, : len(h)] = torch.tensor(h, dtype=torch.long)
        lens[r] = len(h)
    return padded, lens, targets

train_pairs, test, n_items, item_pop = build_dataset()
print(f"n_items={n_items}  n_train_pairs={len(train_pairs)}  n_test_users={len(test)}")
# Expected: n_items=1447  n_train_pairs=53485  n_test_users=938


## Part 1 — Watch vector (user-башня, Covington)

User-башня не имеет собственного id-эмбеддинга: представление юзера собирается из его истории. Шаг, который ты реализуешь, — превращение батча историй переменной длины (с паддингом) в одно представление на юзера, агрегируя эмбеддинги реально просмотренных айтемов. Остальная башня (MLP, item-башня) дана.


In [ ]:
class TwoTower(nn.Module):
    def __init__(self, n_items, dim=64, hidden=128):
        super().__init__()
        self.dim = dim
        self.item_emb = nn.Embedding(n_items, dim)          # общий lookup айтемов
        nn.init.normal_(self.item_emb.weight, std=0.05)
        self.user_mlp = nn.Sequential(                       # user-башня над watch vector
            nn.Linear(dim, hidden), nn.ReLU(),
            nn.Linear(hidden, dim),
        )

    def user_tower(self, hist_emb_mean):
        return self.user_mlp(hist_emb_mean)                 # (B, d)

    def item_tower(self, item_ids):
        return self.item_emb(item_ids)                      # (B, d)

    def watch_vector(self, hist_padded, hist_len):
        """Covington watch vector: одно представление истории на юзера.

        Args:
          hist_padded: (B, L) long — id айтемов истории, справа добитые паддингом
                       (нулями) до общей длины L; реальная длина строки r = hist_len[r].
          hist_len:    (B,) long — число реальных (непаддинговых) айтемов в каждой строке.
        Returns:
          (B, d) float — представление истории каждого юзера в пространстве item-эмбеддингов.
        Note: padding не влияет на результат (история [a,b] и она же, добитая до длины 5, дают одно и то же).
        """
        # YOUR CODE HERE
        raise NotImplementedError


In [ ]:
# Контракт: паддинг не влияет на watch vector; форма (B, d).
torch.manual_seed(0)
_m = TwoTower(n_items)
# строка 0: история [3,7,2]; строка 1: история [5] — добьём паддингом до L=3
padded = torch.tensor([[3,7,2],[5,0,0]], dtype=torch.long)
lens   = torch.tensor([3,1], dtype=torch.long)
wv = _m.watch_vector(padded, lens)
assert wv.shape == (2, _m.dim), f"форма watch vector должна быть (B, d), а не {tuple(wv.shape)}"

# инвариантность к паддингу: та же строка 1, но добитая до L=5 другим мусором
padded2 = torch.tensor([[5,0,0,0,0]], dtype=torch.long)
lens2   = torch.tensor([1], dtype=torch.long)
wv1_short = wv[1]
wv1_long  = _m.watch_vector(padded2, lens2)[0]
assert torch.allclose(wv1_short, wv1_long, atol=1e-6), \
    "паддинг просочился в результат: учитывай только первые hist_len позиций каждой строки"

# истина для строки 0: чистое среднее трёх эмбеддингов
ref0 = _m.item_emb(torch.tensor([3,7,2])).mean(0)
assert torch.allclose(wv[0], ref0, atol=1e-6), "watch vector строки 0 != среднему её эмбеддингов"
print("ok — watch vector корректен и инвариантен к паддингу")


❓ **Inline Question 2.** Covington берёт **среднее** эмбеддингов истории, а не сумму и не последний айтем. Что сломалось бы при сумме (подумай про юзеров с очень разной длиной истории и про L2-нормировку башни ниже)?

**Ваш ответ:** …


## Part 2 — In-batch sampled softmax + logQ-коррекция (вклад Yi 2019)

Ядро статьи. У тебя батч из $B$ пар (юзер, его позитивный айтем). Матрица скоров $B\times B$: скор юзера к айтему. Диагональ — позитивы, остальное — in-batch негативы. Без коррекции эта оценка софтмакса смещена в пользу штрафа популярных айтемов; коррекция вычитает из логитов лог-вероятность попадания айтема в батч, делая оценку несмещённой.

Реализуй обе ветви (с коррекцией и без) одной функцией, переключаемой флагом — дальше мы сведём их в одну абляцию.


In [ ]:
def in_batch_softmax_loss(user_emb, item_emb, log_q, use_correction):
    """In-batch sampled softmax loss для two-tower (контракт Yi 2019).

    Args:
      user_emb: (B, d) — выходы user-башни для B юзеров батча.
      item_emb: (B, d) — выходы item-башни для их B позитивных айтемов.
                Айтем j — позитив юзера j и in-batch негатив всем остальным юзерам.
      log_q:    (B,) — log Q(item_j): лог-вероятность того, что айтем j попал в батч.
      use_correction: bool — применять ли logQ-коррекцию.
    Returns:
      скаляр-тензор — средняя по батчу кросс-энтропия классификации
      «какой из B айтемов батча — позитив данного юзера», где правильный класс
      юзера i есть айтем i (диагональ); дифференцируемый скаляр (по нему .backward()).
    """
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# 1) shape/finiteness/grad
torch.manual_seed(1)
B, d = 6, 8
ue = torch.randn(B, d, requires_grad=True)
ie = torch.randn(B, d)
lq = torch.randn(B)
L = in_batch_softmax_loss(ue, ie, lq, use_correction=True)
assert L.dim() == 0 and torch.isfinite(L), "лосс должен быть конечным скаляром"
L.backward()
assert ue.grad is not None and ue.grad.abs().max() > 0, "градиент не течёт в user_emb — точно ли это CE по батчу?"

# 2) коррекция реально двигает лосс (вклад статьи живой)
torch.manual_seed(2)
ue2 = torch.randn(B, d); ie2 = torch.randn(B, d)
lq2 = torch.tensor([3.0, -3.0, 3.0, -3.0, 3.0, -3.0])   # резко неравномерное Q
L_on  = in_batch_softmax_loss(ue2, ie2, lq2, use_correction=True)
L_off = in_batch_softmax_loss(ue2, ie2, lq2, use_correction=False)
assert not torch.allclose(L_on, L_off), "коррекция не влияет на лосс — вычитаешь ли log_q из логитов нужного айтема?"

# 3) тождество: при равномерном log_q (все равны) коррекция = сдвиг логитов на константу
#    по строке -> softmax инвариантен -> лоссы должны совпасть
lq_const = torch.full((B,), 1.234)
La = in_batch_softmax_loss(ue2, ie2, lq_const, use_correction=True)
Lb = in_batch_softmax_loss(ue2, ie2, lq_const, use_correction=False)
assert torch.allclose(La, Lb, atol=1e-5), \
    "при равномерном log_q коррекция = константный сдвиг по строке и не должна менять softmax/лосс"
print(f"ok — in-batch loss корректен; L_on={L_on.item():.4f} != L_off={L_off.item():.4f}")


❓ **Inline Question 3.** В тесте при **равномерном** $\log Q$ (все айтемы одинаково популярны) коррекция ничего не меняет. Почему это правильное поведение, а не баг? Что это говорит о том, в каком режиме коррекция вообще нужна?

**Ваш ответ:** …


## Part 3 — Стриминговый оценщик $Q$ (Algorithm 1, Yi 2019)

$Q(y)$ — вероятность, что айтем $y$ попадёт в случайный батч — не дано: батчи приходят из живого распределения трафика. Yi оценивает $Q$ на лету по среднему интервалу между появлениями $y$ в потоке шагов: видим часто → маленький средний интервал → большой $\hat Q$.

Реализуй два метода оценщика: онлайн-обновление статистики при наблюдении айтемов на шаге $t$, и выдачу $\log\hat Q$ для запрошенных айтемов. Состояние (`A`, `B`, `seen`, `alpha`) уже заведено в `__init__`.


In [ ]:
class StreamingFreqEstimator:
    """Algorithm 1 (Yi et al. 2019): онлайн-оценка in-batch вероятности Q(item)
    по среднему числу шагов между двумя появлениями айтема в потоке батчей.

    Состояние (per item):
      A[y]    — последний шаг t, на котором айтем y наблюдался (init 0).
      B[y]    — бегущая оценка среднего интервала между появлениями y (init 1).
      seen[y] — наблюдался ли y хоть раз (init False).
      alpha   — скорость экспоненциального сглаживания интервала.
    """

    def __init__(self, n_items, alpha=0.01):
        self.A = np.zeros(n_items)
        self.B = np.ones(n_items)          # init интервал=1 -> Q=1 до первого реального наблюдения
        self.alpha = alpha
        self.seen = np.zeros(n_items, dtype=bool)

    def update(self, item_ids, t):
        """Учесть наблюдение айтемов на шаге t (обновляет состояние in-place).
        Args:    item_ids: (B,) long — наблюдённые айтемы; t: int — номер шага.
        Returns: None — меняет A/B/seen in-place.
        Note: точная форма обновления — Algorithm 1 (см. Теорию Part 3).
        """
        # YOUR CODE HERE
        raise NotImplementedError

    def log_q(self, item_ids):
        """log Q(item) — лог in-batch вероятности айтемов.
        Args:    item_ids: (M,) long — айтемы для оценки.
        Returns: (M,) float32 — log Q(item); конечные значения, Q в (0, 1].
        """
        # YOUR CODE HERE
        raise NotImplementedError


In [ ]:
# Поток: айтем 0 — каждый шаг (частый), айтем 5 — раз в ~10 шагов (редкий).
# alpha побольше, чтобы стриминговая оценка сошлась на коротком синтетическом потоке.
fe = StreamingFreqEstimator(n_items, alpha=0.3)
t = 0
for _ in range(200):
    fe.update([0], t); t += 1
    if t % 10 == 0:
        fe.update([5], t); t += 1

lq = fe.log_q([0, 5])
assert lq.shape == (2,) and lq.dtype == torch.float32, f"log_q: ждём float32 (2,), получили {tuple(lq.shape)} {lq.dtype}"
assert torch.isfinite(lq).all(), "log_q должен быть конечным (clip Q в (0,1], делитель защищён от нуля)"
assert (lq <= 1e-4).all(), "Q — вероятность в (0,1], значит log_q <= 0"
# Observed: log_q(frequent)=-0.272 > log_q(rare)=-2.302
assert lq[0] > lq[1] + 0.5, f"частый айтем обязан получить больший log_q: {lq.tolist()}"

# update реально пишет состояние: A[y] == последний шаг
fe2 = StreamingFreqEstimator(n_items)
fe2.update([4, 9], 17)
assert fe2.A[4] == 17 and fe2.A[9] == 17 and fe2.seen[4] and fe2.seen[9], \
    "после update A[y] должно стать t, а seen[y]=True"
print(f"ok — log_q(frequent)={lq[0]:.3f} > log_q(rare)={lq[1]:.3f}")


❓ **Inline Question 4.** `B` инициализируется единицами (→ $Q=1$) и для впервые встреченного айтема интервал не обновляется. Что это означает для **холодного** айтема, которого почти не видели, на первых батчах — его логит штрафуется сильно или слабо? Хорошо это или плохо для retrieval хвоста?

**Ваш ответ:** …


## Part 4 — Training loop (пишешь сама)

Собери всё: на каждом батче обнови оценщик $Q$, посчитай watch vector → user-башню → item-башню, L2-нормируй обе башни и подели user-логиты на температуру (косинус-retrieval, как в Yi), посчитай in-batch loss и шагни оптимизатором.

Нормировка и температура $\tau=0.07$ даны в комментариях контракта (это обвязка, не ядро) — ядро — порядок «обнови Q на ЭТОМ батче, потом возьми его log_q».


In [ ]:
def train(model, train_pairs, n_items, use_correction, epochs=3, bs=256, lr=1e-2):
    """Обучить two-tower in-batch sampled softmax.
    Args:    model: TwoTower; train_pairs: list[(hist, pos_item)]; n_items: int;
             use_correction: bool — включить logQ; epochs/bs/lr — гиперпараметры.
    Returns: TwoTower — обученная model.
    Компоненты под рукой: collate, freq (оценщик Q), model.watch_vector/user_tower/item_tower,
    in_batch_softmax_loss, opt (Adam); скор косинусный с температурой TEMP. Печатай loss по эпохам, пары перемешивай.
    Note: оценщик Q обязан учесть текущий батч ДО взятия по нему log_q (иначе оценка отстаёт на шаг).
    """
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    freq = StreamingFreqEstimator(n_items)
    TEMP = 0.07
    step = 0
    order = np.arange(len(train_pairs))
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Smoke-train на подвыборке: лосс обязан падать. (Полный прогон — в Part 5.)
torch.manual_seed(0); np.random.seed(0)
sub = train_pairs[:6000]
m_smoke = TwoTower(n_items)
import contextlib, io as _io
buf = _io.StringIO()
with contextlib.redirect_stdout(buf):
    train(m_smoke, sub, n_items, use_correction=True, epochs=3)
import re
losses = [float(x) for x in re.findall(r"loss=([0-9.]+)", buf.getvalue())]
assert len(losses) >= 3, "train должен печатать средний loss по каждой эпохе (3 числа)"
assert losses[-1] < losses[0], f"loss не падает за 3 эпохи: {losses} — порядок update Q / нормировка / температура?"
# grad реально протёк хоть раз
assert any(p.grad is not None for p in m_smoke.parameters()), "ни у одного параметра нет градиента — был ли backward?"
print(f"ok — loss падает {losses[0]:.3f} -> {losses[-1]:.3f}")


## Part 5 — Retrieval + Recall@K (применяем метод)

Теперь сам инференс: для каждого тестового юзера построить его user-вектор из истории, скорить ВСЕ айтемы каталога, **замаскировать уже виденные**, взять top-K и проверить, попал ли спрятанный таргет. Это и есть candidate generation в работе — не проекция, а реальный ретрив по всему каталогу.

`all_item_vectors` (нормированная матрица всех item-векторов) дана.


In [ ]:
@torch.no_grad()
def all_item_vectors(model):
    ids = torch.arange(model.item_emb.num_embeddings)
    return F.normalize(model.item_tower(ids), dim=-1)        # (N, d)

@torch.no_grad()
def recall_at_k(model, test, ks=(10, 50)):
    """Recall@K (== HitRate@K) обученной two-tower на leave-one-out тесте.

    Args:
      test: dict user -> (hist, target), где hist — список id истории,
            target — спрятанный последний айтем.
      ks:   какие K считать.
    Returns:
      dict {K: recall@K}, где recall@K — доля тестовых юзеров (с непустой
      историей), у которых спрятанный target попал в top-K выданных моделью айтемов.
      Юзеров с пустой историей не считай; уже виденные айтемы из выдачи исключи.
    Note: представление юзера строй той же user-башней/нормировкой, что в обучении (иначе train/eval рассогласуются).
    """
    item_vecs = all_item_vectors(model)
    # YOUR CODE HERE
    raise NotImplementedError

def recall_at_k_popularity(test, item_pop, ks=(10, 50)):
    """Бейзлайн: рекомендуем всем самые частые айтемы (минус виденные)."""
    order = np.argsort(-item_pop)
    hits = {k: 0 for k in ks}; n = 0
    for u, (hist, target) in test.items():
        if len(hist) == 0:
            continue
        seen = set(hist)
        rec = [int(i) for i in order if int(i) not in seen]
        n += 1
        for k in ks:
            if target in rec[:k]:
                hits[k] += 1
    return {k: hits[k] / max(n, 1) for k in ks}


In [ ]:
# Проверяем recall@k на маленькой обученной модели + контракт маскирования.
torch.manual_seed(0); np.random.seed(0)
m_eval = TwoTower(n_items)
import contextlib, io as _io
with contextlib.redirect_stdout(_io.StringIO()):
    train(m_eval, train_pairs[:6000], n_items, use_correction=True, epochs=2)

r = recall_at_k(m_eval, test, ks=(10, 50))
assert set(r.keys()) == {10, 50}, "recall_at_k должен вернуть dict по ks"
assert 0.0 <= r[10] <= 1.0 and 0.0 <= r[50] <= 1.0, "recall — доля в [0,1]"
assert r[50] >= r[10], "R@50 не может быть меньше R@10 (top-50 включает top-10)"

# контракт маскирования: виденные айтемы не должны попадать в выдачу.
# берём юзера и проверяем, что его таргет, если он СЛУЧАЙНО в истории, не «засчитан» через историю
item_vecs = all_item_vectors(m_eval)
u0 = next(u for u, (h, t) in test.items() if len(h) > 0)
h0, _ = test[u0]
padded = torch.tensor(h0, dtype=torch.long).unsqueeze(0)
lens = torch.tensor([len(h0)], dtype=torch.long)
uvec = F.normalize(m_eval.user_tower(m_eval.watch_vector(padded, lens)), dim=-1)
scores = (uvec @ item_vecs.t()).squeeze(0).clone()
scores[torch.tensor(h0, dtype=torch.long)] = -1e9
top = torch.topk(scores, 50).indices.tolist()
assert not (set(h0) & set(top)), "виденные айтемы просочились в top-K — замаскируй историю перед topk"
print(f"ok — recall@k работает; на под-обученной модели R@10={r[10]:.4f} R@50={r[50]:.4f}")


### Полный прогон — обучаем обе модели и popularity-бейзлайн

Готовая ячейка: обучаем two-tower **с** коррекцией и **без** (абляция), считаем три набора Recall@K. Несколько минут на CPU.


In [ ]:
torch.manual_seed(0); np.random.seed(0)

print("training two-tower WITH logQ correction (Yi 2019) ...")
model_c = TwoTower(n_items).to(DEVICE)
train(model_c, train_pairs, n_items, use_correction=True, epochs=3)

print("training two-tower WITHOUT correction (ablation) ...")
model_u = TwoTower(n_items).to(DEVICE)
train(model_u, train_pairs, n_items, use_correction=False, epochs=3)

rec_c   = recall_at_k(model_c, test)
rec_u   = recall_at_k(model_u, test)
rec_pop = recall_at_k_popularity(test, item_pop)

print("\n--- Recall@K (== HitRate@K) ---")
print(f"popularity baseline : R@10={rec_pop[10]:.4f}  R@50={rec_pop[50]:.4f}")
print(f"two-tower (no corr) : R@10={rec_u[10]:.4f}  R@50={rec_u[50]:.4f}")
print(f"two-tower (+logQ)   : R@10={rec_c[10]:.4f}  R@50={rec_c[50]:.4f}")
# Expected (seed 0): pop 0.0501/0.1684 | no-corr 0.0416/0.2004 | +logQ 0.0832/0.2772


In [ ]:
# Контракт результата: метод бьёт бейзлайн, и коррекция помогает.
# Пороги с запасом откалиброваны по реальному прогону (seed 0/7).
print(f"+logQ   R@10={rec_c[10]:.4f}  R@50={rec_c[50]:.4f}")
print(f"no-corr R@10={rec_u[10]:.4f}  R@50={rec_u[50]:.4f}")
print(f"pop     R@10={rec_pop[10]:.4f}  R@50={rec_pop[50]:.4f}")

assert rec_c[50] >= 0.25, f"R@50 метода {rec_c[50]:.4f} ниже ожидаемого ~0.277 — недообучение/баг ретрива"
assert rec_c[10] >= 0.07, f"R@10 метода {rec_c[10]:.4f} ниже ожидаемого ~0.083"
assert rec_c[50] > rec_pop[50] + 0.08, \
    f"two-tower должен заметно бить popularity по R@50 (наблюдаемый разрыв ~0.11): {rec_c[50]:.4f} vs {rec_pop[50]:.4f}"
# абляция: logQ-коррекция обязана помочь (вклад статьи) — наблюдаемо 0.083 vs 0.042
assert rec_c[10] > rec_u[10], \
    f"logQ-коррекция должна поднять R@10 над абляцией: {rec_c[10]:.4f} (corr) vs {rec_u[10]:.4f} (no-corr)"
print("ok — метод бьёт popularity, и logQ-коррекция помогает")


## Ключевая фигура — три системы рядом

Headline-результат из твоих функций. Сравни с тем, что заявляет Yi 2019: на их корпусе logQ-коррекция стабильно улучшает retrieval-метрику; абляция «без коррекции» проседает именно на популярном/головном трафике. У нас тот же знак эффекта на MovieLens.


In [ ]:
import matplotlib.pyplot as plt
labels = ["popularity", "two-tower\n(no corr)", "two-tower\n(+logQ)"]
r10 = [rec_pop[10], rec_u[10], rec_c[10]]
r50 = [rec_pop[50], rec_u[50], rec_c[50]]
x = np.arange(3); w = 0.38
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - w/2, r10, w, label="Recall@10")
ax.bar(x + w/2, r50, w, label="Recall@50")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("Recall@K (== HitRate@K)")
ax.set_title("Candidate generation на MovieLens-100k (leave-one-out)")
ax.legend(); ax.grid(True, axis="y", alpha=0.3)
for xi, v in zip(x - w/2, r10): ax.text(xi, v + 0.003, f"{v:.3f}", ha="center", fontsize=8)
for xi, v in zip(x + w/2, r50): ax.text(xi, v + 0.003, f"{v:.3f}", ha="center", fontsize=8)
plt.tight_layout(); plt.show()
print("в статье logQ-коррекция стабильно > абляции; у нас тот же знак: +logQ бьёт и no-corr, и popularity")


## Пощупать фишку — где именно коррекция выигрывает

Абляция уже встроена в прогон: единственная разница между `model_c` и `model_u` — флаг `use_correction`. Это и есть естественный контраст, в котором видно преимущество. Посмотрим на эффект там, где теория его обещает: на **популярных** айтемах. Без коррекции in-batch негативы переучивают модель «гасить» хиты; коррекция возвращает им справедливый логит.


In [ ]:
# Берём топ-популярные таргеты теста и считаем R@50 отдельно на них:
# где коррекция и должна отыгрывать смещение in-batch негативов.
pop_rank = {int(i): r for r, i in enumerate(np.argsort(-item_pop))}
pop_thresh = np.percentile([pop_rank[t] for (_, (_, t)) in test.items()], 25)  # топ-25% по популярности таргета

def recall_subset(model, only_popular):
    iv = all_item_vectors(model); hit = n = 0
    for u, (hist, target) in test.items():
        if len(hist) == 0: continue
        is_pop = pop_rank[int(target)] <= pop_thresh
        if is_pop != only_popular: continue
        padded = torch.tensor(hist, dtype=torch.long).unsqueeze(0)
        lens = torch.tensor([len(hist)], dtype=torch.long)
        uvec = F.normalize(model.user_tower(model.watch_vector(padded, lens)), dim=-1)
        sc = (uvec @ iv.t()).squeeze(0).clone()
        sc[torch.tensor(hist, dtype=torch.long)] = -1e9
        n += 1; hit += int(target in torch.topk(sc, 50).indices.tolist())
    return hit / max(n, 1), n

for only_pop in (True, False):
    rc, n = recall_subset(model_c, only_pop); ru, _ = recall_subset(model_u, only_pop)
    seg = "популярные таргеты" if only_pop else "хвостовые таргеты"
    print(f"{seg:20s} (n={n:4d}): +logQ R@50={rc:.4f} | no-corr R@50={ru:.4f} | дельта={rc-ru:+.4f}")


❓ **Inline Question 5 (узнавание/перенос).** Представь свою задачу: retrieval по корпусу с **тяжёлым степенным хвостом популярности** и in-batch негативами (товары, видео, музыка). В какой новой ситуации ты первым делом потянешься за logQ-коррекцией — и по какому симптому в метриках поймёшь, что смещение in-batch негативов уже бьёт по качеству?

**Ваш ответ:** …


## Debug-area

Хрупкие узлы и как их ловить до полного обучения:

- **watch_vector** — forward на dummy-входе ДО обучения: `TwoTower(n_items).watch_vector(padded, lens)` должен дать `(B, d)` и быть инвариантным к паддингу (тест Part 1). Самый частый баг — делишь на `L` (паддинговую длину), а не на `hist_len`.
- **in_batch_softmax_loss** — проверь на `B=2`: логиты `2×2`, метки `[0,1]`. Руками посчитай CE при единичных эмбеддингах. Если коррекция «не двигает лосс» — ты вычитаешь `log_q` не по тому измерению (нужен айтем-столбец, не юзер-строка).
- **StreamingFreqEstimator** — порядок в train: `update` на текущем шаге → потом `log_q`. Если перепутать, текущий батч не учитывается и оценка отстаёт.
- **train** — если лосс не падает: забыл `opt.zero_grad()`; или не поделил user-логиты на температуру (косинусы в `[-1,1]`, softmax почти плоский, градиент мизерный); или нормируешь не по `dim=-1`.
- **recall_at_k** — если R@K подозрительно высок, проверь маскирование истории (виденные айтемы лезут в top-K) и что user-вектор строишь тем же путём, что в обучении.


## Going deeper (bonus, без баллов)

1. **Чувствительность к $\alpha$.** Прогони оценщик $Q$ с `alpha` $\in\{0.001, 0.01, 0.1\}$ и посмотри на разброс $\log Q$ и на итоговый Recall. Где компромисс между «быстро адаптируется» и «шумит»?
2. **Температура $\tau$.** Свип `TEMP` $\in\{0.03, 0.07, 0.2\}$. Как меняется острота softmax и метрика?
3. **nDCG@K вместо Recall@K.** Один held-out таргет делает Recall == HitRate; добавь несколько held-out (last-N) и посчитай nDCG — изменится ли знак выигрыша коррекции?
4. **Uniform-негативы вместо in-batch.** Замени in-batch негативы на сэмпл из всего каталога. Нужна ли тогда logQ-коррекция? (Свяжи с Inline Question 3.)

> Лог эксперимента: …


## SPOILERS — полные решения (открывай ТОЛЬКО когда уверена / застряла)

Сначала попробуй сама. Подсказки в докстрингах и в Debug-area обычно хватает.

<details>
<summary>Part 1 — watch_vector</summary>

```python
def watch_vector(self, hist_padded, hist_len):
    emb = self.item_emb(hist_padded)                                  # (B, L, d)
    mask = (torch.arange(hist_padded.size(1))[None, :] < hist_len[:, None])
    emb = emb * mask.unsqueeze(-1)
    summed = emb.sum(1)
    return summed / hist_len.clamp(min=1).unsqueeze(-1).float()
```
</details>

<details>
<summary>Part 2 — in_batch_softmax_loss</summary>

```python
def in_batch_softmax_loss(user_emb, item_emb, log_q, use_correction):
    logits = user_emb @ item_emb.t()                 # (B, B)
    if use_correction:
        logits = logits - log_q.unsqueeze(0)         # вычесть log Q по столбцу-айтему
    labels = torch.arange(logits.size(0), device=logits.device)
    return F.cross_entropy(logits, labels)
```
</details>

<details>
<summary>Part 3 — StreamingFreqEstimator.update / log_q</summary>

```python
def update(self, item_ids, t):
    for y in item_ids:
        if self.seen[y]:
            self.B[y] = (1 - self.alpha) * self.B[y] + self.alpha * (t - self.A[y])
        self.A[y] = t
        self.seen[y] = True

def log_q(self, item_ids):
    q = 1.0 / np.maximum(self.B[item_ids], 1e-6)
    q = np.clip(q, 1e-6, 1.0)
    return torch.tensor(np.log(q), dtype=torch.float32)
```
</details>

<details>
<summary>Part 4 — train (тело после обвязки)</summary>

```python
    for ep in range(epochs):
        np.random.shuffle(order)
        total = 0.0; nb = 0
        for s in range(0, len(order), bs):
            idx = order[s : s + bs]
            batch = [train_pairs[j] for j in idx]
            padded, lens, targets = collate(batch)
            freq.update(targets.tolist(), step); step += 1
            log_q = freq.log_q(targets.tolist())
            wv = model.watch_vector(padded, lens)
            u  = F.normalize(model.user_tower(wv), dim=-1) / TEMP
            it = F.normalize(model.item_tower(targets), dim=-1)
            loss = in_batch_softmax_loss(u, it, log_q, use_correction)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item(); nb += 1
        print(f"  epoch {ep+1}: loss={total/nb:.4f}")
    return model
```
</details>

<details>
<summary>Part 5 — recall_at_k (тело после all_item_vectors)</summary>

```python
    hits = {k: 0 for k in ks}; n = 0
    for u, (hist, target) in test.items():
        if len(hist) == 0:
            continue
        padded = torch.tensor(hist, dtype=torch.long).unsqueeze(0)
        lens = torch.tensor([len(hist)], dtype=torch.long)
        wv = model.watch_vector(padded, lens)
        uvec = F.normalize(model.user_tower(wv), dim=-1)
        scores = (uvec @ item_vecs.t()).squeeze(0).clone()
        scores[torch.tensor(hist, dtype=torch.long)] = -1e9    # маскируем виденное
        n += 1
        for k in ks:
            if target in torch.topk(scores, k).indices.tolist():
                hits[k] += 1
    return {k: hits[k] / max(n, 1) for k in ks}
```
</details>


## Рефлексия

**Что украсть в свою работу.** Приём «оцени смещение сэмплирования и вычти его лог из логита» — общий: всякий раз, когда обучающие негативы приходят из распределения, отличного от целевого (а это почти всегда, когда негативы дешёвые), несмещённость возвращается дешёвой аддитивной поправкой к логиту, а не дорогим пересэмплированием. Стриминговая оценка неизвестного $Q$ по интервалам между появлениями — отдельный переносимый трюк для любого онлайн-счётчика частот без второго прохода.

**В какой НОВОЙ ситуации этот метод — правильный инструмент.** Любой large-corpus retrieval с in-batch / popularity-пропорциональными негативами и тяжёлым хвостом: товарные рекомендации, поиск, dense retrieval для RAG (in-batch негативы там тоже смещены к частым пассажам). Симптом, по которому узнаёшь задачу: метрика на голове проседает относительно популярности, а на хвосте подозрительно высока — модель переучилась гасить хиты.

**Что осталось непонятным → следующий узел.** Как logQ-коррекция взаимодействует с hard-negative mining (если негативы выбраны НЕ случайно, а по сложности — какое $Q$ тогда вычитать?). И: full sampled-softmax с обучаемым temperature/mixed-negative — отдельная ветка дерева.
